# Session 7 - deployable proxies + free text


**GPU T4 x2, ~3 h.** Two analyses that need only the suspect image.

* **Deployable proxies**: blur / noise / shuffle / selfcheck perturbations of region k,
  validated against the true FS. Each proxy's **floor** -- its effect on an *authentic*
  frame -- is measured as well, and a proxy with a positive floor is disqualified
  regardless of its correlation.
* **Free-text explanations**: the model explains itself in a sentence, which is mapped
  onto the same vocabulary so that FS can be recomputed for the stated explanation.

In [ ]:
SESSION = "S7 proxy"

# ============================== CONFIG ==============================
MODEL      = "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct"
QUANT      = "fp16"
MAX_PIXELS = 384 * 384
SPLIT      = "test"
SEED       = 0

PROXIES        = "blur,noise,shuffle,selfcheck"
PROXY_SAMPLES  = 600
PROXY_BUDGET   = 210

FREE_TEXT_SAMPLES = 300
FREE_TEXT_BUDGET  = 120

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils")
KU.gpu_report()
INDEX = KU.find_parsed_index()
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
RESUME = ",".join(d for d in KU.find_run_dirs() if "run_" in d)
n_gpu = max(1, KU.n_gpus())

In [ ]:
# ==================== PROXY CONDITIONS ====================
cmds, envs, logs = [], [], []
for i in range(n_gpu):
    logs.append(f"{OUT}/logs/proxy_{i}.log")
    envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
    cmds.append(
        f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
        f'--detector "{MODEL}" --out "{OUT}/run_proxy" --tag proxy '
        f'--split {SPLIT} --limit-samples {PROXY_SAMPLES} --seed {SEED} '
        f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
        f'--max-pixels {MAX_PIXELS} --proxy "{PROXIES}" --splice-floor '
        f'--vocab {VOCAB} --time-budget-min {PROXY_BUDGET}'
        + (f' --resume-from "{RESUME}"' if RESUME else ""))
KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== FREE TEXT ====================
cmds, envs, logs = [], [], []
for i in range(n_gpu):
    logs.append(f"{OUT}/logs/text_{i}.log")
    envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
    cmds.append(
        f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
        f'--detector "{MODEL}" --out "{OUT}/run_text" --tag freetext '
        f'--split {SPLIT} --limit-samples {FREE_TEXT_SAMPLES} --seed {SEED} '
        f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
        f'--max-pixels {MAX_PIXELS} --free-text --vocab {VOCAB} '
        f'--time-budget-min {FREE_TEXT_BUDGET}')
KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== ANALYSE ====================
KU.sh(f'{sys.executable} -m ccaudit.m11_proxy --raw "{OUT}/run_proxy" '
      f'--out "{OUT}/proxy" --vocab {VOCAB} --split {SPLIT}', check=False)
KU.sh(f'{sys.executable} -m ccaudit.m12_text_regions --raw "{OUT}/run_text" '
      f'--out "{OUT}/text" --vocab {VOCAB}', check=False)

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s7-proxy`.
2. In the proxy ranking, a deployable proxy must have both a high correlation
   with the true FS **and** a floor whose confidence interval includes zero.
   If every proxy is disqualified by its floor, that is recorded as the
   outcome of the analysis: it is the same defect that disqualifies an
   inpainting operator."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)